In [1]:
import numpy as np
# Importowanie najwazniejszych komponentow Sionna RT
from sionna.rt import PathSolver
from sionna_env import setup_scene, update_object_position, save_reference_data, save_reference_data_to_file
import ofdm_config as cfg
from IPython.display import clear_output

scene_file = "./OpenStreetMap/kampus_AGH_test_all/Kampus_AGH_B9_test_all.xml"

scene, tx, rx = setup_scene(scene_file, cfg.fc, cfg.rx_rows, cfg.rx_cols)

dron_target = scene.get("dron_target")
# print(f'Pozycja drona: {dron_target.position}')

samochod_target_b = scene.get("samochod_target_b")
# print(f'Pozycja samochodu_b: {samochod_target_b.position}')

samochod_target_d = scene.get("samochod_target_d")
# print(f'Pozycja samochodu_d: {samochod_target_d.position}')

czlowiek_target = scene.get("czlowiek_target")
# print(f'Pozycja czlowieka: {czlowiek_target.position}')

print(f'Pomyślnie załadowano scenę: {scene_file}')


Pomyślnie załadowano scenę: ./OpenStreetMap/kampus_AGH_test_all/Kampus_AGH_B9_test_all.xml


In [4]:
import mitsuba as mi

def check_los(scene, radar_pos, target_obj, z_offset=1.0, margin=1.0):
    p_radar = np.asarray(radar_pos, dtype=np.float32).reshape(3)
    p_target = np.asarray(target_obj.position, dtype=np.float32).reshape(3)
    p_target[2] += z_offset

    direction = p_target - p_radar
    distance = float(np.linalg.norm(direction))

    ray = mi.Ray3f(
        o=mi.Point3f(p_radar),
        d=mi.Vector3f(direction / distance)
    )
    ray.maxt = distance - margin
    return not bool(scene.mi_scene.ray_test(ray))

In [5]:
positions = scene.rx_array.positions(cfg.wavelength)

try:
    antenna_positions_sionna = np.stack(
        [
            np.asarray(positions.x),
            np.asarray(positions.y),
            np.asarray(positions.z),
        ],
        axis=-1,
    )
except AttributeError:
    antenna_positions_sionna = np.asarray(positions)

print("shape:", antenna_positions_sionna.shape)
print(antenna_positions_sionna)

np.save("antenna_positions_4x4.npy", antenna_positions_sionna)

shape: (16, 3)
[[ 0.         -0.00803571  0.00803571]
 [ 0.         -0.00803571  0.00267857]
 [ 0.         -0.00803571 -0.00267857]
 [ 0.         -0.00803571 -0.00803571]
 [ 0.         -0.00267857  0.00803571]
 [ 0.         -0.00267857  0.00267857]
 [ 0.         -0.00267857 -0.00267857]
 [ 0.         -0.00267857 -0.00803571]
 [ 0.          0.00267857  0.00803571]
 [ 0.          0.00267857  0.00267857]
 [ 0.          0.00267857 -0.00267857]
 [ 0.          0.00267857 -0.00803571]
 [ 0.          0.00803571  0.00803571]
 [ 0.          0.00803571  0.00267857]
 [ 0.          0.00803571 -0.00267857]
 [ 0.          0.00803571 -0.00803571]]


In [6]:
num_steps = 200
frame_time = 0.1

ref_drone = []
ref_car_B = []
ref_car_D = []
ref_pedestrian = []

for i in range(num_steps):
    t = (i + 1) * frame_time

    # 1. Ruch drona
    if i == 150:
        dron_target.position = [0.0, 0.0, -50.0]
        dron_vel = np.array([0, 0, 0], dtype=float)
        update_object_position(dron_target, dron_vel, frame_time)
    elif i < 150:
        if i > 120:
            dron_vel = np.array([13, 7, 0], dtype=float)
        elif i > 80:
            dron_vel = np.array([9, 13, 0.5], dtype=float)
        elif i > 50:
            dron_vel = np.array([15, 7, 0], dtype=float)
        elif i > 30:
            dron_vel = np.array([8, 14, 0], dtype=float)
        else:
            dron_vel = np.array([5, 13, -4], dtype=float)
        update_object_position(dron_target, dron_vel, frame_time)

        row = save_reference_data(dron_target, rx, i, t, "drone")
        row["is_los"] = check_los(scene, rx.position, dron_target, z_offset=0.0, margin=0.5)
        ref_drone.append(row)

    # 2. Ruch samochodu B
    if i == 195:
        samochod_target_b.position = [0.0, 0.0, -50.0]
        samochod_target_b_vel = np.array([0, 0, 0], dtype=float)
        update_object_position(samochod_target_b, samochod_target_b_vel, frame_time)
    elif i < 195:
        if i > 120:
            samochod_target_b_vel = np.array([11, 0, 0], dtype=float)
        else:
            samochod_target_b_vel = np.array([9, -0.2, 0], dtype=float)
        update_object_position(samochod_target_b, samochod_target_b_vel, frame_time)

        row = save_reference_data(samochod_target_b, rx, i, t, "car_B")
        row["is_los"] = check_los(scene, rx.position, samochod_target_b, z_offset=1.0, margin=2.0)
        ref_car_B.append(row)

    # 3. Ruch samochodu D
    if i <= 190:
        if i > 180:
            samochod_target_d_vel = np.array([-6.4, 0, 0], dtype=float)
        elif i > 170:
            samochod_target_d_vel = np.array([-5, 0, 0], dtype=float)
        elif i > 80:
            samochod_target_d_vel = np.array([0, 8.8, 0], dtype=float)
        elif i > 50:
            samochod_target_d_vel = np.array([14, 0, 0], dtype=float)
        else:
            samochod_target_d_vel = np.array([16, 0, 0], dtype=float)
    else:
        samochod_target_d_vel = np.array([0, 0, 0], dtype=float)

    update_object_position(samochod_target_d, samochod_target_d_vel, frame_time)
    row = save_reference_data(samochod_target_d, rx, i, t, "car_D")
    row["is_los"] = check_los(scene, rx.position, samochod_target_d, z_offset=1.0, margin=2.0)
    ref_car_D.append(row)

    # 4. Ruch człowieka
    if i > 150:
        czlowiek_target_vel = np.array([0, 1.5, 0], dtype=float)
    elif i > 100:
        czlowiek_target_vel = np.array([-1, 1.3, 0], dtype=float)
    else:
        czlowiek_target_vel = np.array([0, 1.3, 0], dtype=float)

    update_object_position(czlowiek_target, czlowiek_target_vel, frame_time)
    row = save_reference_data(czlowiek_target, rx, i, t, "pedestrian")
    row["is_los"] = check_los(scene, rx.position, czlowiek_target, z_offset=1.0, margin=1.0)
    ref_pedestrian.append(row)

save_reference_data_to_file(ref_drone, "ref_drone.csv")
save_reference_data_to_file(ref_car_B, "ref_car_B.csv")
save_reference_data_to_file(ref_car_D, "ref_car_D.csv")
save_reference_data_to_file(ref_pedestrian, "ref_pedestrian.csv")